In [ ]:

%load_ext autoreload
%autoreload 2
import time
import sys
import numpy as np
import Components as comp
import socket
import random
import Measurement_functions as mf
if socket.gethostname() == "ph-photonbec5":
    sys.path.append("D:/Control/PythonPackages/")
    sys.path.append("D:/Control/PiezoController/")






In [ ]:

%load_ext autoreload
%autoreload 2
import time
import sys
import numpy as np
import Components as comp
import socket
import matplotlib.pyplot as plt
import Measurement_functions as mf
if socket.gethostname() == "ph-photonbec5":
    sys.path.append("D:/Control/PythonPackages/")
    sys.path.append("D:/Control/PiezoController/")


laser = comp.Toptica_Laser()
plot_flag_g2 = False
binning= 7
# You can change this sets the resolution as well
offset = 0
tacq = 15000 # Measurement time in millisec, you can change this
# power_list = np.append(np.append(np.arange(65, 58, 0.25)/1000,np.arange(58, 61, 0.1)/1000),np.arange(61,65,0.5)/1000)
# power_list = np.append(np.arange(16, 19, 0.25)/1000,np.arange(19, 24, 0.05)/1000)
#power_list = np.arange(17,22,0.02)/1000
power_list = np.arange(31,36,0.05)/1000 #938
position = 20
ND1 = 23
ND2 = 23.7

for power in power_list:
    print('Power = ', power*1000, 'mW')
    laser.set(power)
    time.sleep(1)

    # if np.round(power*1000,2) == ND1:
    #     print('Please put ND1')
    #     input()
    # if np.round(power*1000,2) == ND2:
    #     print('Please put ND2')
    #     input()


    scan = mf.get_hh_data(binning, offset, tacq,plot_flag_g2)
    data = scan[0]
    resolution = scan[1]
    mf.save_hh_data(data, resolution, tacq,power*1000,position)
    counts = data/np.mean(data[350:500])
    x = np.arange(0,len(counts),1)*0.128
    plt.figure(figsize=(10, 6))
    plt.plot(x,counts, label='Counts (Channel 0)')
    plt.title('Counts vs. Time Bin (Channel 0)')
    plt.xlabel('Time Bin')
    plt.ylabel('Counts')
    plt.xlim(0,80)
    plt.ylim(np.min(counts[0:350]-0.025),np.max(counts)+0.025)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


    time.sleep(10)
laser.set(0)

In [ ]:
"WITH POWER METER"
%load_ext autoreload
%autoreload 2
import time
import sys
import numpy as np
import Components as comp
import socket
import matplotlib.pyplot as plt
import Measurement_functions as mf
import threading
if socket.gethostname() == "ph-photonbec5":
    sys.path.append("D:/Control/PythonPackages/")
    sys.path.append("D:/Control/PiezoController/")

powermeter = comp.Thor_PowerMeter(power_meter_usb_name='USB0::0x1313::0x8078::P0034379::INSTR', num_power_readings=100, bs_factor=1, wavelength=950)
laser = comp.Toptica_Laser()
plot_flag_g2 = False
binning= 7
# You can change this sets the resolution as well
offset = 0
tacq = 15000 # Measurement time in millisec, you can change this
# power_list = np.append(np.append(np.arange(65, 58, 0.25)/1000,np.arange(58, 61, 0.1)/1000),np.arange(61,65,0.5)/1000)
# power_list = np.append(np.arange(16, 19, 0.25)/1000,np.arange(19, 24, 0.05)/1000)
#power_list = np.append(np.append(np.arange(12, 13, 0.05)/1000,np.arange(13, 16, 0.02)/1000),np.arange(16,18.5,0.05)/1000)
#power_list = np.append(np.append(np.arange(10.5, 11.5, 0.05)/1000,np.arange(11.5, 13.5, 0.02)/1000),np.arange(13.5,14.5,0.05)/1000) #950
#power_list = np.append(np.append(np.arange(12.5, 13.5, 0.05)/1000,np.arange(13.5, 15, 0.02)/1000),np.arange(15,16.5,0.05)/1000) #945
#power_list = np.append(np.append(np.arange(23.5, 24.5, 0.05)/1000,np.arange(24.5, 26, 0.02)/1000),np.arange(26,27,0.05)/1000) #942
#power_list = np.append(np.append(np.arange(17.5, 18.5, 0.05)/1000,np.arange(18.5, 20, 0.02)/1000),np.arange(20,21,0.05)/1000) #942

power_list = np.arange(50,60,0.05)/1000 #938
position = 20
ND1 = 23
ND2 = 23.7


for power in power_list:
    print('Power = ', power*1000, 'mW')
    laser.set(power)
    time.sleep(1)
    measured_power = []
    measuring = True

    def measure_power():
        while measuring:
            reading = 10**6 * powermeter.take_power_reading()
            measured_power.append(reading)
            time.sleep(0.1)

    def get_g2():
        scan = mf.get_hh_data(binning, offset, tacq, plot_flag_g2)
        data = scan[0]
        resolution = scan[1]
        counts = data / np.mean(data[350:500])
        x = np.arange(0, len(counts), 1) * 0.128
        plt.figure(figsize=(10, 6))
        plt.plot(x, counts, label='Counts (Channel 0)')
        plt.title('Counts vs. Time Bin (Channel 0)')
        plt.xlabel('Time Bin')
        plt.ylabel('Counts')
        plt.xlim(0, 80)
        plt.ylim(np.min(counts[0:350] - 0.025), np.max(counts) + 0.025)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.show()
        return data,resolution,tacq,power*1000


    power_thread = threading.Thread(target=measure_power)
    power_thread.start()

    data,resolution,tacq,power = get_g2()
    mf.save_hh_data(data, resolution, tacq, power, np.mean(measured_power))
    measuring = False
    power_thread.join()

    if measured_power:
        print("Average Measured Power: {:.3f} µW".format(np.mean(measured_power)))
    else:
        print("No power readings were taken.")
    laser.set(0)
    time.sleep(30)
laser.set(0)

In [ ]:
print(1/((984.5/911.8)-1))